# DentaScribe — Integration & Backend
## Role 5 — Koroush | Individual Role Notebook

| | |
|---|---|
| **Author** | Koroush (Role 5 — Integration & Backend) |
| **Course** | Deep Learning Group Project |
| **Scope** | End-to-end pipeline connecting all models + filing system |
| **Team notebook** | See `dentascribe_final_v3.ipynb` for all roles combined |

---

### What this notebook does
1. **Section 1** — Architecture overview and model loading
2. **Section 2** — Audio transcription (Whisper)
3. **Section 3** — Named entity recognition (DistilBERT NER)
4. **Section 4** — Form classification and autofill (DistilBERT Classifier)
5. **Section 5** — Filing system (JSON storage by patient ID + date)
6. **Section 6** — End-to-end pipeline demo
7. **Section 7** — Error handling and edge cases

> **Input:** Audio file + patient ID  
> **Output:** Auto-filled JSON form saved under `filing_system/{patient_id}/{date}/`


In [ ]:
!pip install transformers torch librosa soundfile -q

## Section 1 — Architecture Overview and Model Loading

The DentaScribe pipeline connects four components in sequence:

```
Audio File
    │
    ▼
Whisper ASR  (Role 1 — Andy)
    │  transcript (str)
    ▼
DistilBERT NER  (Role 2 — Ali)
    │  entities: [{word, label, score}]
    ▼
DistilBERT Form Classifier  (Role 4 — Iva)
    │  form: {form_type, confidence, diagnosis, medication, ...}
    ▼
Filing System  (Role 5 — Koroush)
    │  JSON saved to filing_system/{patient_id}/{date}/
    ▼
Streamlit UI  (Role 6 — Aparna)
```


In [ ]:
def load_whisper(model_path='openai/whisper-base'):
    """Load the Whisper ASR processor and model.

    Args:
        model_path (str): HuggingFace model ID or local path.
            Defaults to 'openai/whisper-base'.

    Returns:
        tuple[WhisperProcessor, WhisperForConditionalGeneration]:
            Loaded processor and model in eval mode.
    """
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    processor = WhisperProcessor.from_pretrained(model_path)
    model     = WhisperForConditionalGeneration.from_pretrained(model_path)
    model.eval()
    print(f'[Whisper] Loaded from: {model_path}')
    return processor, model

def load_ner(model_path='dental_ner_model'):
    """Load the DistilBERT NER pipeline.

    Args:
        model_path (str): Path to fine-tuned NER model folder.
            Defaults to 'dental_ner_model'.

    Returns:
        transformers.Pipeline: Token-classification pipeline.
    """
    from transformers import pipeline
    ner = pipeline('token-classification', model=model_path, aggregation_strategy='simple')
    print(f'[NER] Loaded from: {model_path}')
    return ner

def load_form_classifier(model_path='iva_form_classifier/final'):
    """Load the DistilBERT form classification model.

    Args:
        model_path (str): Path to fine-tuned classifier folder.
            Defaults to 'iva_form_classifier/final'.

    Returns:
        tuple[AutoTokenizer, AutoModelForSequenceClassification]:
            Loaded tokenizer and model in eval mode.
    """
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()
    print(f'[Form Classifier] Loaded from: {model_path}')
    return tokenizer, model

print('[Section 1] Model loader functions defined.')


## Section 2 — Audio Transcription (Whisper)

In [ ]:
def transcribe_audio(audio_path, processor, model):
    """Transcribe a WAV or MP3 audio file using Whisper.

    Loads audio, converts to mono, resamples to 16 kHz, normalises
    amplitude, and runs Whisper inference.

    Args:
        audio_path (str): Path to the input audio file.
        processor (WhisperProcessor): Loaded Whisper feature extractor.
        model (WhisperForConditionalGeneration): Loaded Whisper model.

    Returns:
        str: Plain-text transcript of the audio.
    """
    import torch, numpy as np, librosa
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    audio    = audio / (np.max(np.abs(audio)) + 1e-9)
    inputs   = processor(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        ids = model.generate(inputs['input_features'])
    transcript = processor.batch_decode(ids, skip_special_tokens=True)[0]
    print(f'[Transcription] {transcript}')
    return transcript

print('[Section 2] Transcription function defined.')


## Section 3 — Named Entity Recognition

In [ ]:
def extract_entities(transcript, ner_pipeline):
    """Extract dental entities from a transcript using the NER model.

    Merges BERT subword tokens and filters short fragments.
    Returns entities labelled Disease (diagnoses) or Chemical (medications).

    Args:
        transcript (str): Plain-text dental consultation transcript.
        ner_pipeline (transformers.Pipeline): Loaded NER pipeline.

    Returns:
        list[dict]: Entities with keys 'word', 'label', 'score'.
    """
    entities = ner_pipeline(transcript.lower())
    merged   = []
    for ent in entities:
        word, label, score = ent.get('word',''), ent.get('entity_group', ent.get('entity','')), ent.get('score',0)
        if word.startswith('##') and merged:
            merged[-1]['word']  += word[2:]
            merged[-1]['score']  = round((merged[-1]['score'] + score) / 2, 4)
        else:
            merged.append({'word': word.strip(), 'label': label, 'score': round(score, 4)})
    cleaned = [e for e in merged if len(e['word'].strip()) > 2]
    print(f'[NER] Entities found: {cleaned}')
    return cleaned

print('[Section 3] NER function defined.')


## Section 4 — Form Classification and Autofill

In [ ]:
def classify_and_fill_form(transcript, entities, tokenizer, model):
    """Classify the dental form type and autofill fields from NER entities.

    Args:
        transcript (str): Full consultation transcript.
        entities (list[dict]): NER entities from extract_entities.
        tokenizer (AutoTokenizer): Form classifier tokenizer.
        model (AutoModelForSequenceClassification): Form classifier model.

    Returns:
        dict: Completed form with keys form_type, confidence, diagnosis,
            medication, original_text, status, timestamp.
    """
    import torch
    from datetime import datetime
    inputs = tokenizer(transcript, return_tensors='pt', truncation=True,
                       max_length=128, padding='max_length')
    with torch.no_grad():
        logits = model(**inputs).logits
    probs      = torch.softmax(logits, dim=1)
    pred_id    = probs.argmax().item()
    confidence = probs[0][pred_id].item()
    form_type  = model.config.id2label[pred_id]
    diagnosis  = [e['word'] for e in entities if 'Disease' in e['label'] or 'DISEASE' in e['label']]
    medication = [e['word'] for e in entities if 'Chemical' in e['label'] or 'CHEM' in e['label']]
    form = {
        'form_type':     form_type,
        'confidence':    round(confidence, 4),
        'diagnosis':     diagnosis  if diagnosis  else 'not detected',
        'medication':    medication if medication else 'not detected',
        'original_text': transcript,
        'status':        'ready_for_review',
        'timestamp':     datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }
    print(f'[Form] Classified as: {form_type} ({confidence:.1%})')
    return form

print('[Section 4] Form classification function defined.')


## Section 5 — Filing System

Forms are saved as JSON files under:
```
filing_system/
  {patient_id}/
    {YYYY-MM-DD}/
      {form_type}_{HH-MM-SS}.json
```


In [ ]:
import os, json
from datetime import datetime

def save_to_filing_system(patient_id, form, base_dir='filing_system'):
    """Save a completed dental form as a JSON file.

    Creates the directory structure automatically if it does not exist.

    Args:
        patient_id (str): Unique patient identifier (e.g. 'P001').
        form (dict): Completed form dict from classify_and_fill_form.
        base_dir (str): Root directory for the filing system.
            Defaults to 'filing_system'.

    Returns:
        str: Full path to the saved JSON file.
    """
    folder = os.path.join(base_dir, patient_id, datetime.now().strftime('%Y-%m-%d'))
    os.makedirs(folder, exist_ok=True)
    filename = f"{form['form_type']}_{datetime.now().strftime('%H-%M-%S')}.json"
    filepath = os.path.join(folder, filename)
    with open(filepath, 'w') as f:
        json.dump(form, f, indent=2)
    print(f'[Filing] Saved to: {filepath}')
    return filepath

# ── Demo: save a sample form ─────────────────────────────────────────
sample_form = {
    'form_type': 'cleaning_form', 'confidence': 0.97,
    'diagnosis': ['gingivitis'], 'medication': ['chlorhexidine rinse'],
    'original_text': 'Patient has mild gingivitis. Scaling performed.',
    'status': 'ready_for_review', 'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}
path = save_to_filing_system('P001', sample_form)
print(f'Sample form saved: {path}')


## Section 6 — End-to-End Pipeline Demo

In [ ]:
def run_pipeline(audio_path, patient_id,
                 whisper_processor, whisper_model,
                 ner_pipe, form_tokenizer, form_model):
    """Run the full DentaScribe pipeline on a single audio file.

    Args:
        audio_path (str): Path to the input audio file.
        patient_id (str): Patient identifier for the filing system.
        whisper_processor (WhisperProcessor): Loaded Whisper processor.
        whisper_model (WhisperForConditionalGeneration): Loaded Whisper model.
        ner_pipe (transformers.Pipeline): Loaded NER pipeline.
        form_tokenizer (AutoTokenizer): Form classifier tokenizer.
        form_model (AutoModelForSequenceClassification): Form classifier model.

    Returns:
        dict: Keys are transcript, entities, form, saved_path.
    """
    print('=' * 50)
    print('DentaScribe Pipeline Starting')
    print('=' * 50)
    transcript = transcribe_audio(audio_path, whisper_processor, whisper_model)
    entities   = extract_entities(transcript, ner_pipe)
    form       = classify_and_fill_form(transcript, entities, form_tokenizer, form_model)
    saved_path = save_to_filing_system(patient_id, form)
    print(f'\nPipeline complete.')
    print(f'  Form type : {form["form_type"]}')
    print(f'  Confidence: {form["confidence"]:.1%}')
    print(f'  Saved to  : {saved_path}')
    print('=' * 50)
    return {'transcript': transcript, 'entities': entities,
            'form': form, 'saved_path': saved_path}

# ── To run the full pipeline, load models and call: ──────────────────
# whisper_processor, whisper_model = load_whisper('openai/whisper-base')
# ner_pipe                         = load_ner('dental_ner_model')
# form_tokenizer, form_model       = load_form_classifier('iva_form_classifier/final')
# result = run_pipeline('sample.wav', 'P001',
#                       whisper_processor, whisper_model,
#                       ner_pipe, form_tokenizer, form_model)
print('[Section 6] Pipeline function defined.')


## Section 7 — Error Handling and Edge Cases

In [ ]:
def run_pipeline_safe(audio_path, patient_id,
                      whisper_processor, whisper_model,
                      ner_pipe, form_tokenizer, form_model):
    """Run the pipeline with full error handling for production use.

    Catches and logs errors at each stage without crashing the application.

    Args:
        audio_path (str): Path to the input audio file.
        patient_id (str): Patient identifier.
        whisper_processor: Loaded Whisper processor.
        whisper_model: Loaded Whisper model.
        ner_pipe: Loaded NER pipeline.
        form_tokenizer: Form classifier tokenizer.
        form_model: Form classifier model.

    Returns:
        dict: Result dict with a 'status' key ('success' or 'error').
    """
    result = {'status': 'error', 'patient_id': patient_id, 'audio_path': audio_path}
    try:
        result['transcript'] = transcribe_audio(audio_path, whisper_processor, whisper_model)
    except Exception as e:
        result['error'] = f'Transcription failed: {e}'
        return result
    try:
        result['entities'] = extract_entities(result['transcript'], ner_pipe)
    except Exception as e:
        result['entities'] = []
        print(f'[Warning] NER failed: {e} — continuing with empty entities.')
    try:
        result['form'] = classify_and_fill_form(
            result['transcript'], result['entities'], form_tokenizer, form_model)
        result['saved_path'] = save_to_filing_system(patient_id, result['form'])
        result['status']     = 'success'
    except Exception as e:
        result['error'] = f'Form classification or filing failed: {e}'
    return result

print('[Section 7] Safe pipeline function defined.')
print('Integration notebook complete.')
